### Pipeline
```
UCF101 (Kaggle Dataset) → Preprocessing → R3D-18 Fine-tuning → best_model.pth
```

In [24]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU!')

Device : cuda
GPU    : Tesla T4
Memory : 15.6 GB


In [25]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    if files:
        print(f'  Files: {files[:3]}')
    break  

/kaggle/input


In [26]:
import os

INPUT_DIR    = '/kaggle/input'
DATASET_ROOT = None
for root, dirs, files in os.walk(INPUT_DIR):
    avi_files = [f for f in files if f.endswith('.avi')]
    if avi_files:
        DATASET_ROOT = root.split('/train')[0].split('/val')[0].split('/test')[0]
        break
if DATASET_ROOT:
    print(f'UCF101 found at: {DATASET_ROOT}')
    print(f'Contents: {os.listdir(DATASET_ROOT)}')
else:
    print('No .avi files found anywhere in /kaggle/input')
    for root, dirs, files in os.walk(INPUT_DIR):
        print(root)

UCF101 found at: /kaggle/input/datasets/matthewjansen/ucf101-action-recognition
Contents: ['val.csv', 'val', 'train.csv', 'test.csv', 'test', 'train']


In [27]:
!pip install -q scikit-learn opencv-python-headless

In [28]:
%%writefile /kaggle/working/preprocess.py
import cv2
import torch
import numpy as np

def sample_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = max(total_frames // num_frames, 1)
    frames = []
    for i in range(num_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else np.zeros((112, 112, 3), dtype=np.uint8))
    return frames

def resize_frames(frames, size=(112, 112)):
    resized = [cv2.resize(f, size) for f in frames]
    return np.array(resized)

def normalize_and_tensor(frames_array):
    frames_array = frames_array.astype(np.float32) / 255.0
    return torch.from_numpy(frames_array)

def format_for_r3d(tensor):
    tensor = tensor.permute(3, 0, 1, 2)
    tensor = tensor.unsqueeze(0)
    return tensor

Overwriting /kaggle/working/preprocess.py


In [29]:
%%writefile /kaggle/working/dataset.py
import sys
sys.path.insert(0, '/kaggle/working')

import torch
from torch.utils.data import Dataset
from preprocess import sample_frames, resize_frames, normalize_and_tensor, format_for_r3d

class UCF101Dataset(Dataset):
    def __init__(self, video_paths, labels):
        self.video_paths = video_paths
        self.labels = labels

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        path  = self.video_paths[idx]
        label = self.labels[idx]
        frames       = sample_frames(path)
        resized      = resize_frames(frames)
        tensor       = normalize_and_tensor(resized)
        final_tensor = format_for_r3d(tensor).squeeze(0)
        return final_tensor, label

Overwriting /kaggle/working/dataset.py


In [30]:
%%writefile /kaggle/working/model.py
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights

def get_r3d_model(num_classes):
    model = r3d_18(weights=R3D_18_Weights.DEFAULT)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    return model

Overwriting /kaggle/working/model.py


In [31]:
%%writefile /kaggle/working/data_loader.py
import os
import sys
import pandas as pd
from torch.utils.data import DataLoader
sys.path.insert(0, '/kaggle/working')
from dataset import UCF101Dataset

def get_dataloaders(dataset_root, batch_size=16, num_workers=2):
    train_df = pd.read_csv(os.path.join(dataset_root, 'train.csv'))
    val_df   = pd.read_csv(os.path.join(dataset_root, 'val.csv'))
    all_classes  = sorted(train_df['label'].unique().tolist())
    class_to_idx = {cls: idx for idx, cls in enumerate(all_classes)}

    def build_lists(df):
        paths, labels = [], []
        for _, row in df.iterrows():
            rel  = row['clip_path'].lstrip('/')
            full = os.path.join(dataset_root, rel)
            if os.path.exists(full):
                paths.append(full)
                labels.append(class_to_idx[row['label']])
        return paths, labels

    train_paths, train_labels = build_lists(train_df)
    val_paths,   val_labels   = build_lists(val_df)

    if len(train_paths) == 0:
        raise ValueError('No training videos found!')

    train_loader = DataLoader(UCF101Dataset(train_paths, train_labels),
                              batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(UCF101Dataset(val_paths, val_labels),
                              batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, class_to_idx

Overwriting /kaggle/working/data_loader.py


In [32]:
import sys
sys.path.insert(0, '/kaggle/working')

for mod in ['data_loader', 'dataset', 'preprocess', 'model']:
    if mod in sys.modules:
        del sys.modules[mod]

from data_loader import get_dataloaders
train_loader, val_loader, class_to_idx = get_dataloaders(DATASET_ROOT, batch_size=16)
clips, labels = next(iter(train_loader))

print(f'\nBatch shape : {clips.shape}')  
print(f'Labels      : {labels.tolist()}')


Batch shape : torch.Size([16, 3, 16, 112, 112])
Labels      : [94, 1, 7, 76, 71, 42, 26, 97, 66, 6, 83, 38, 6, 24, 28, 16]


In [36]:
import os, sys, json, time, shutil
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from model import get_r3d_model
from data_loader import get_dataloaders

sys.path.insert(0, '/kaggle/working')

for mod in ['model', 'data_loader', 'dataset', 'preprocess']:
    if mod in sys.modules:
        del sys.modules[mod]

EPOCHS     = 15
BATCH_SIZE = 16      
LR         = 1e-4
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = '/kaggle/working/best_model.pth'
JSON_PATH  = '/kaggle/working/class_names.json'

print(f'Device : {DEVICE}')
print(f'Epochs : {EPOCHS}')
print(f'Batch  : {BATCH_SIZE}')

train_loader, val_loader, class_to_idx = get_dataloaders(
    DATASET_ROOT, batch_size=BATCH_SIZE
)
num_classes = len(class_to_idx)

with open(JSON_PATH, 'w') as f:
    json.dump(class_to_idx, f, indent=2)
print(f'Saved class_names.json — {num_classes} classes')

model     = get_r3d_model(num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for clips, lbls in loader:
            clips, lbls = clips.to(DEVICE), lbls.to(DEVICE)
            outputs = model(clips)
            loss    = criterion(outputs, lbls)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * clips.size(0)
            correct    += (outputs.argmax(1) == lbls).sum().item()
            total      += clips.size(0)
    return total_loss / total, correct / total * 100

best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(val_loader,   training=False)
    scheduler.step()
    elapsed = time.time() - t0
    print(f'Epoch [{epoch:02d}/{EPOCHS}] '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}%  |  '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.1f}%  '
          f'[{elapsed/60:.1f} min]')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'✔ Best model saved — val_acc={val_acc:.1f}%')

print(f'\n🎉 Training complete! Best val accuracy: {best_val_acc:.1f}%')
print(f'Model saved to: {MODEL_PATH}')

✅ model.py written
✅ preprocess.py written
✅ dataset.py written
✅ data_loader.py written
Device : cuda
Epochs : 15
Batch  : 16
[data_loader] Classes : 101
[data_loader] Train   : 10055 videos
[data_loader] Val     : 1673 videos
Saved class_names.json — 101 classes
Epoch [01/15] Train Loss: 1.4383 Acc: 76.0%  |  Val Loss: 0.2488 Acc: 94.4%  [17.2 min]
  ✔ Best model saved — val_acc=94.4%
Epoch [02/15] Train Loss: 0.1712 Acc: 97.4%  |  Val Loss: 0.1127 Acc: 97.3%  [17.2 min]
  ✔ Best model saved — val_acc=97.3%
Epoch [03/15] Train Loss: 0.0578 Acc: 98.8%  |  Val Loss: 0.1049 Acc: 96.8%  [17.7 min]
Epoch [04/15] Train Loss: 0.0490 Acc: 98.6%  |  Val Loss: 0.1162 Acc: 96.5%  [17.3 min]
Epoch [05/15] Train Loss: 0.0406 Acc: 98.8%  |  Val Loss: 0.0982 Acc: 97.3%  [17.3 min]
Epoch [06/15] Train Loss: 0.0225 Acc: 99.0%  |  Val Loss: 0.0721 Acc: 97.3%  [17.4 min]
Epoch [07/15] Train Loss: 0.0190 Acc: 99.0%  |  Val Loss: 0.0686 Acc: 97.1%  [17.6 min]
Epoch [08/15] Train Loss: 0.0171 Acc: 99.1%  

In [39]:
import json, torch, os, sys
sys.path.insert(0, '/kaggle/working')
from model import get_r3d_model
from preprocess import sample_frames, resize_frames, normalize_and_tensor, format_for_r3d

for mod in ['model', 'preprocess']:
    if mod in sys.modules: 
        del sys.modules[mod]

with open('/kaggle/working/class_names.json') as f:
    class_to_idx = json.load(f)
idx_to_class = {v: k for k, v in class_to_idx.items()}

model = get_r3d_model(len(class_to_idx))
model.load_state_dict(torch.load('/kaggle/working/best_model.pth', map_location='cpu'))
model.eval()

sample_class = list(class_to_idx.keys())[0]
sample_dir   = os.path.join(DATASET_ROOT, 'train', sample_class)
sample_video = os.path.join(sample_dir, os.listdir(sample_dir)[0])
print(f'Testing on : {sample_video}')
print(f'True class : {sample_class}')

frames = sample_frames(sample_video)
tensor = format_for_r3d(normalize_and_tensor(resize_frames(frames)))

with torch.no_grad():
    outputs = model(tensor)
    probs   = torch.softmax(outputs, dim=1)[0]
    top3    = probs.topk(3)

print('\nTop-3 Predictions:')
for prob, idx in zip(top3.values, top3.indices):
    print(f'{idx_to_class[idx.item()]:30s} {prob.item()*100:.1f}%')

✅ model.py written
✅ preprocess.py written
Testing on : /kaggle/input/datasets/matthewjansen/ucf101-action-recognition/train/ApplyEyeMakeup/v_ApplyEyeMakeup_g19_c01.avi
True class : ApplyEyeMakeup

Top-3 Predictions:
  ApplyEyeMakeup                 100.0%
  ApplyLipstick                  0.0%
  Haircut                        0.0%


In [40]:
import os

model_size = os.path.getsize('/kaggle/working/best_model.pth') / 1e6
print(f'✅ best_model.pth    : {model_size:.1f} MB')
print(f'✅ class_names.json  : ready')
print()
print('To download:')
print('  → Look at the Output tab on the right panel')
print('  → Click the download icon next to each file')
print('  → Save to: C:/Users/sreej/OneDrive/Desktop/webcam2/')

✅ best_model.pth    : 133.0 MB
✅ class_names.json  : ready

To download:
  → Look at the Output tab on the right panel
  → Click the download icon next to each file
  → Save to: C:/Users/sreej/OneDrive/Desktop/webcam2/
